In [ ]:
```python
!pip -q install requests beautifulsoup4 lxml

import os
import time
import warnings
import requests
import pandas as pd

from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from urllib3.exceptions import InsecureRequestWarning

from google.colab import drive

# ============================================================
# GOOGLE DRIVE
# ============================================================

drive.mount("/content/drive")

# ============================================================
# SETTINGS
# ============================================================

START_ID = 730_000
END_ID = 939_999

PART_NAME = "part_04"

WORKERS = 5
TIMEOUT = 20
SAVE_EVERY = 5_000
CHUNK_SIZE = WORKERS * 10

# ============================================================
# OUTPUT DIRECTORY
# ============================================================

BASE_DIR = "/content/drive/MyDrive/alriyadh_1million"

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    PART_NAME
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

PROGRESS_FILE = os.path.join(
    OUTPUT_DIR,
    "progress.txt"
)

# ============================================================
# WARNINGS
# ============================================================

warnings.simplefilter(
    "ignore",
    InsecureRequestWarning
)

# ============================================================
# RESUME SYSTEM
# ============================================================

if os.path.exists(PROGRESS_FILE):

    try:

        with open(PROGRESS_FILE, "r") as f:
            saved_id = int(f.read().strip())

        CURRENT_ID = saved_id + 1

        print(
            f"RESUMING FROM ID: {CURRENT_ID:,}"
        )

    except Exception:

        CURRENT_ID = START_ID

        print(
            f"STARTING FROM ID: {CURRENT_ID:,}"
        )

else:

    CURRENT_ID = START_ID

    print(
        f"STARTING FROM ID: {CURRENT_ID:,}"
    )

if CURRENT_ID > END_ID:

    print("=" * 80)
    print("THIS PART IS ALREADY FINISHED.")
    print("=" * 80)

    raise SystemExit

# ============================================================
# SESSION
# ============================================================

def create_session():

    session = requests.Session()

    retry = Retry(
        total=2,
        connect=2,
        read=2,
        backoff_factor=1,
        status_forcelist=[
            429,
            500,
            502,
            503,
            504
        ],
        allowed_methods=[
            "GET"
        ]
    )

    adapter = HTTPAdapter(
        max_retries=retry,
        pool_connections=WORKERS,
        pool_maxsize=WORKERS
    )

    session.mount(
        "http://",
        adapter
    )

    session.mount(
        "https://",
        adapter
    )

    session.headers.update({
        "User-Agent":
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/151.0.0.0 Safari/537.36",

        "Accept-Language":
            "ar-SA,ar;q=0.9,en;q=0.8"
    })

    return session

# ============================================================
# SCRAPE ONE ARTICLE
# ============================================================

def scrape_article(article_id):

    session = create_session()

    url = f"https://www.alriyadh.com/{article_id}"

    try:

        response = session.get(
            url,
            timeout=TIMEOUT,
            verify=False
        )

        if response.status_code != 200:
            return None

        soup = BeautifulSoup(
            response.text,
            "lxml"
        )

        # ====================================================
        # TITLE
        # ====================================================

        title = ""

        og_title = soup.find(
            "meta",
            property="og:title"
        )

        if og_title and og_title.get("content"):

            title = og_title["content"].strip()

        if not title:

            h1 = soup.find("h1")

            if h1:

                title = h1.get_text(
                    " ",
                    strip=True
                )

        # ====================================================
        # CATEGORY
        # ====================================================

        category = ""

        active_items = soup.select(
            "li.active"
        )

        categories = []

        for li in active_items:

            a = li.find("a")

            if not a:
                continue

            text = a.get_text(
                " ",
                strip=True
            )

            if text:
                categories.append(text)

        if len(categories) >= 2:

            category = categories[1]

        elif len(categories) == 1:

            category = categories[0]

        # ====================================================
        # COUNTRY
        # ====================================================

        country = ""

        country_candidates = soup.select(
            '[class*="country"],'
            '[class*="Country"],'
            '[itemprop="contentLocation"]'
        )

        for element in country_candidates:

            text = element.get_text(
                " ",
                strip=True
            )

            if text:

                country = text
                break

        # ====================================================
        # DATE
        # ====================================================

        date = ""

        date_candidates = soup.select(
            '[itemprop="datePublished"],'
            'time,'
            '[class*="date"],'
            '[class*="Date"]'
        )

        for element in date_candidates:

            value = (
                element.get("datetime")
                or element.get_text(
                    " ",
                    strip=True
                )
            )

            if value:

                date = value
                break

        # ====================================================
        # VALID ARTICLE
        # ====================================================

        if not title:
            return None

        return {
            "url": url,
            "title": title,
            "category": category,
            "country": country,
            "date": date
        }

    except Exception:

        return None

# ============================================================
# SAVE BATCH
# ============================================================

def save_batch(data, batch_number):

    if not data:
        return

    df = pd.DataFrame(data)

    output_file = os.path.join(
        OUTPUT_DIR,
        f"batch_{batch_number:05d}.csv"
    )

    df.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )

    print()
    print("=" * 80)
    print(f"SAVED: {output_file}")
    print(f"ROWS: {len(df):,}")
    print("=" * 80)

# ============================================================
# FIND NEXT BATCH NUMBER
# ============================================================

existing_batches = [
    filename
    for filename in os.listdir(OUTPUT_DIR)
    if filename.startswith("batch_")
    and filename.endswith(".csv")
]

batch_number = 1

if existing_batches:

    numbers = []

    for filename in existing_batches:

        try:

            number = int(
                filename
                .replace("batch_", "")
                .replace(".csv", "")
            )

            numbers.append(number)

        except Exception:
            pass

    if numbers:

        batch_number = max(numbers) + 1

# ============================================================
# START
# ============================================================

print()
print("=" * 80)
print("ALRIYADH - DIRECT ARTICLE SCRAPER")
print("=" * 80)

print(
    f"PART:       {PART_NAME}"
)

print(
    f"START ID:   {CURRENT_ID:,}"
)

print(
    f"END ID:     {END_ID:,}"
)

print(
    f"WORKERS:    {WORKERS}"
)

print(
    f"SAVE EVERY: {SAVE_EVERY:,}"
)

print(
    f"CHUNK SIZE: {CHUNK_SIZE}"
)

print("=" * 80)

start_time = time.time()

batch = []

completed_ids = CURRENT_ID - 1

current = CURRENT_ID

# ============================================================
# MAIN LOOP
# ============================================================

while current <= END_ID:

    chunk_end = min(
        current + CHUNK_SIZE - 1,
        END_ID
    )

    ids = range(
        current,
        chunk_end + 1
    )

    with ThreadPoolExecutor(
        max_workers=WORKERS
    ) as executor:

        futures = {
            executor.submit(
                scrape_article,
                article_id
            ): article_id
            for article_id in ids
        }

        for future in as_completed(futures):

            try:

                result = future.result()

                if result:

                    batch.append(result)

            except Exception:

                pass

    # ========================================================
    # UPDATE PROGRESS
    # ========================================================

    completed_ids = chunk_end

    with open(
        PROGRESS_FILE,
        "w"
    ) as f:

        f.write(
            str(completed_ids)
        )

    # ========================================================
    # SAVE EVERY 5,000
    # ========================================================

    while len(batch) >= SAVE_EVERY:

        save_data = batch[:SAVE_EVERY]

        batch = batch[SAVE_EVERY:]

        save_batch(
            save_data,
            batch_number
        )

        batch_number += 1

    # ========================================================
    # PROGRESS
    # ========================================================

    elapsed = (
        time.time()
        - start_time
    )

    checked = (
        completed_ids
        - CURRENT_ID
        + 1
    )

    rate = (
        checked / elapsed
        if elapsed > 0
        else 0
    )

    print(
        f"Progress: "
        f"{completed_ids:,}/"
        f"{END_ID:,} | "
        f"Checked: {checked:,} | "
        f"Collected pending: "
        f"{len(batch):,} | "
        f"Rate: {rate:.2f} IDs/sec"
    )

    current = chunk_end + 1

# ============================================================
# SAVE REMAINING
# ============================================================

if batch:

    save_batch(
        batch,
        batch_number
    )

# ============================================================
# FINISHED
# ============================================================

elapsed = (
    time.time()
    - start_time
)

print()
print("=" * 80)
print("PART FINISHED")
print("=" * 80)

print(
    f"PART:       {PART_NAME}"
)

print(
    f"LAST ID:    {END_ID:,}"
)

print(
    f"TIME:       {elapsed / 3600:.2f} hours"
)

print(
    f"OUTPUT:     {OUTPUT_DIR}"
)

print("=" * 80)



SyntaxError: invalid syntax (1722810908.py, line 1)

In [ ]:
!pip -q install requests beautifulsoup4 lxml

import os
import time
import warnings
import requests
import pandas as pd

from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from urllib3.exceptions import InsecureRequestWarning
from google.colab import drive


# ============================================================
# GOOGLE DRIVE
# ============================================================

drive.mount("/content/drive")


# ============================================================
# SETTINGS - PART 04
# ============================================================

START_ID = 730_000
END_ID = 939_999

PART_NAME = "part_04"

WORKERS = 5
TIMEOUT = 20

SAVE_EVERY = 5_000

CHUNK_SIZE = WORKERS * 10


# ============================================================
# OUTPUT DIRECTORY
# ============================================================

BASE_DIR = "/content/drive/MyDrive/alriyadh_1million"

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    PART_NAME
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

PROGRESS_FILE = os.path.join(
    OUTPUT_DIR,
    "progress.txt"
)


# ============================================================
# WARNINGS
# ============================================================

warnings.simplefilter(
    "ignore",
    InsecureRequestWarning
)


# ============================================================
# RESUME
# ============================================================

if os.path.exists(PROGRESS_FILE):

    try:

        with open(PROGRESS_FILE, "r") as f:
            saved_id = int(f.read().strip())

        CURRENT_ID = saved_id + 1

        print(
            f"RESUMING FROM ID: {CURRENT_ID:,}"
        )

    except Exception:

        CURRENT_ID = START_ID

        print(
            f"STARTING FROM ID: {CURRENT_ID:,}"
        )

else:

    CURRENT_ID = START_ID

    print(
        f"STARTING FROM ID: {CURRENT_ID:,}"
    )


if CURRENT_ID > END_ID:

    print("=" * 80)
    print("THIS PART IS ALREADY FINISHED.")
    print("=" * 80)

    raise SystemExit


# ============================================================
# SESSION
# ============================================================

def create_session():

    session = requests.Session()

    retry = Retry(
        total=2,
        connect=2,
        read=2,
        backoff_factor=1,
        status_forcelist=[
            429,
            500,
            502,
            503,
            504
        ],
        allowed_methods=["GET"]
    )

    adapter = HTTPAdapter(
        max_retries=retry,
        pool_connections=WORKERS,
        pool_maxsize=WORKERS
    )

    session.mount(
        "http://",
        adapter
    )

    session.mount(
        "https://",
        adapter
    )

    session.headers.update({
        "User-Agent":
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/151.0.0.0 Safari/537.36",

        "Accept-Language":
            "ar-SA,ar;q=0.9,en;q=0.8"
    })

    return session


# ============================================================
# SCRAPE ARTICLE
# ============================================================

def scrape_article(article_id):

    session = create_session()

    url = f"https://www.alriyadh.com/{article_id}"

    try:

        response = session.get(
            url,
            timeout=TIMEOUT,
            verify=False
        )

        if response.status_code != 200:
            return None

        soup = BeautifulSoup(
            response.text,
            "lxml"
        )


        # ====================================================
        # TITLE
        # ====================================================

        title = ""

        og_title = soup.find(
            "meta",
            property="og:title"
        )

        if og_title and og_title.get("content"):

            title = og_title["content"].strip()


        if not title:

            h1 = soup.find("h1")

            if h1:

                title = h1.get_text(
                    " ",
                    strip=True
                )


        # ====================================================
        # CATEGORY
        # ====================================================

        category = ""

        active_items = soup.select(
            "li.active"
        )

        categories = []

        for li in active_items:

            a = li.find("a")

            if not a:
                continue

            text = a.get_text(
                " ",
                strip=True
            )

            if text:

                categories.append(text)


        if len(categories) >= 2:

            category = categories[1]

        elif len(categories) == 1:

            category = categories[0]


        # ====================================================
        # COUNTRY
        # ====================================================

        country = ""

        country_candidates = soup.select(
            '[class*="country"],'
            '[class*="Country"],'
            '[itemprop="contentLocation"]'
        )

        for element in country_candidates:

            text = element.get_text(
                " ",
                strip=True
            )

            if text:

                country = text
                break


        # ====================================================
        # DATE
        # ====================================================

        date = ""

        date_candidates = soup.select(
            '[itemprop="datePublished"],'
            'time,'
            '[class*="date"],'
            '[class*="Date"]'
        )

        for element in date_candidates:

            value = (
                element.get("datetime")
                or element.get_text(
                    " ",
                    strip=True
                )
            )

            if value:

                date = value
                break


        # ====================================================
        # VALID ARTICLE
        # ====================================================

        if not title:
            return None


        return {
            "url": url,
            "title": title,
            "category": category,
            "country": country,
            "date": date
        }


    except Exception:

        return None


# ============================================================
# SAVE BATCH
# ============================================================

def save_batch(data, batch_number):

    if not data:
        return

    df = pd.DataFrame(data)

    output_file = os.path.join(
        OUTPUT_DIR,
        f"batch_{batch_number:05d}.csv"
    )

    df.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )

    print()
    print("=" * 80)
    print(f"SAVED: {output_file}")
    print(f"ROWS: {len(df):,}")
    print("=" * 80)


# ============================================================
# FIND NEXT BATCH NUMBER
# ============================================================

existing_batches = [
    filename
    for filename in os.listdir(OUTPUT_DIR)
    if filename.startswith("batch_")
    and filename.endswith(".csv")
]

batch_number = 1

if existing_batches:

    numbers = []

    for filename in existing_batches:

        try:

            number = int(
                filename
                .replace("batch_", "")
                .replace(".csv", "")
            )

            numbers.append(number)

        except Exception:

            pass

    if numbers:

        batch_number = max(numbers) + 1


# ============================================================
# START
# ============================================================

print()
print("=" * 80)
print("ALRIYADH - DIRECT ARTICLE SCRAPER")
print("=" * 80)

print(f"PART:       {PART_NAME}")
print(f"START ID:   {CURRENT_ID:,}")
print(f"END ID:     {END_ID:,}")
print(f"WORKERS:    {WORKERS}")
print(f"SAVE EVERY: {SAVE_EVERY:,}")
print(f"CHUNK SIZE: {CHUNK_SIZE}")

print("=" * 80)


start_time = time.time()

batch = []

completed_ids = CURRENT_ID - 1

current = CURRENT_ID


# ============================================================
# MAIN LOOP
# ============================================================

while current <= END_ID:

    chunk_end = min(
        current + CHUNK_SIZE - 1,
        END_ID
    )

    ids = range(
        current,
        chunk_end + 1
    )


    with ThreadPoolExecutor(
        max_workers=WORKERS
    ) as executor:

        futures = {
            executor.submit(
                scrape_article,
                article_id
            ): article_id
            for article_id in ids
        }


        for future in as_completed(futures):

            try:

                result = future.result()

                if result:

                    batch.append(result)

            except Exception:

                pass


    # ========================================================
    # UPDATE PROGRESS
    # ========================================================

    completed_ids = chunk_end

    with open(
        PROGRESS_FILE,
        "w"
    ) as f:

        f.write(
            str(completed_ids)
        )


    # ========================================================
    # SAVE EVERY 5,000
    # ========================================================

    while len(batch) >= SAVE_EVERY:

        save_data = batch[:SAVE_EVERY]

        batch = batch[SAVE_EVERY:]

        save_batch(
            save_data,
            batch_number
        )

        batch_number += 1


    # ========================================================
    # PROGRESS
    # ========================================================

    elapsed = time.time() - start_time

    checked = (
        completed_ids
        - CURRENT_ID
        + 1
    )

    rate = (
        checked / elapsed
        if elapsed > 0
        else 0
    )

    print(
        f"Progress: "
        f"{completed_ids:,}/"
        f"{END_ID:,} | "
        f"Checked: {checked:,} | "
        f"Collected pending: "
        f"{len(batch):,} | "
        f"Rate: {rate:.2f} IDs/sec"
    )


    current = chunk_end + 1


# ============================================================
# SAVE REMAINING
# ============================================================

if batch:

    save_batch(
        batch,
        batch_number
    )


# ============================================================
# FINISHED
# ============================================================

elapsed = time.time() - start_time

print()
print("=" * 80)
print("PART FINISHED")
print("=" * 80)

print(f"PART:       {PART_NAME}")
print(f"LAST ID:    {END_ID:,}")
print(f"TIME:       {elapsed / 3600:.2f} hours")
print(f"OUTPUT:     {OUTPUT_DIR}")

print("=" * 80)

Mounted at /content/drive
STARTING FROM ID: 730,000

ALRIYADH - DIRECT ARTICLE SCRAPER
PART:       part_04
START ID:   730,000
END ID:     939,999
WORKERS:    5
SAVE EVERY: 5,000
CHUNK SIZE: 50
Progress: 730,049/939,999 | Checked: 50 | Collected pending: 50 | Rate: 4.11 IDs/sec
Progress: 730,099/939,999 | Checked: 100 | Collected pending: 95 | Rate: 4.16 IDs/sec
Progress: 730,149/939,999 | Checked: 150 | Collected pending: 145 | Rate: 4.15 IDs/sec
Progress: 730,199/939,999 | Checked: 200 | Collected pending: 194 | Rate: 4.17 IDs/sec
Progress: 730,249/939,999 | Checked: 250 | Collected pending: 243 | Rate: 4.15 IDs/sec
Progress: 730,299/939,999 | Checked: 300 | Collected pending: 293 | Rate: 4.14 IDs/sec
Progress: 730,349/939,999 | Checked: 350 | Collected pending: 342 | Rate: 4.15 IDs/sec
Progress: 730,399/939,999 | Checked: 400 | Collected pending: 390 | Rate: 4.18 IDs/sec
Progress: 730,449/939,999 | Checked: 450 | Collected pending: 433 | Rate: 4.16 IDs/sec
Progress: 730,499/939,999 

KeyboardInterrupt: 

In [ ]:
# حفظ كل البيانات المعلقة قبل الإيقاف
import os
import pandas as pd

SAVE_DIR = "/content/drive/MyDrive/alriyadh_1million/emergency_save"
os.makedirs(SAVE_DIR, exist_ok=True)

if len(pending) > 0:
    # رقم ملف جديد حتى لا نستبدل حفظًا سابقًا
    existing = [
        f for f in os.listdir(SAVE_DIR)
        if f.startswith("emergency_") and f.endswith(".csv")
    ]
    number = len(existing) + 1

    path = os.path.join(SAVE_DIR, f"emergency_{number:04d}.csv")

    pd.DataFrame(pending).to_csv(
        path,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"تم حفظ {len(pending):,} صف")
    print(f"المسار: {path}")
else:
    print("لا توجد بيانات pending للحفظ.")

print("الآن يمكنك إيقاف الخلية بأمان.")

NameError: name 'pending' is not defined

In [ ]:
# ============================================================
# EMERGENCY SAVE - SAVE CURRENT BATCH
# ============================================================

import os
import pandas as pd

if batch:
    recovery_file = os.path.join(
        OUTPUT_DIR,
        "EMERGENCY_SAVE.csv"
    )

    df = pd.DataFrame(batch)

    df.to_csv(
        recovery_file,
        index=False,
        encoding="utf-8-sig"
    )

    print("=" * 80)
    print("EMERGENCY SAVE COMPLETED")
    print(f"ROWS SAVED: {len(df):,}")
    print(f"FILE: {recovery_file}")
    print("=" * 80)

else:
    print("لا توجد بيانات غير محفوظة في batch.")

NameError: name 'batch' is not defined

In [ ]:
import os

OUTPUT_DIR = "/content/drive/MyDrive/alriyadh_1million/part_04"
PROGRESS_FILE = os.path.join(OUTPUT_DIR, "progress.txt")

print("=" * 70)

# آخر ID محفوظ
if os.path.exists(PROGRESS_FILE):
    with open(PROGRESS_FILE, "r") as f:
        saved_id = f.read().strip()

    print("LAST SAVED ID:", saved_id)
else:
    print("progress.txt غير موجود")

print()
print("CSV FILES:")

files = sorted(
    f for f in os.listdir(OUTPUT_DIR)
    if f.endswith(".csv")
)

for f in files:
    size = os.path.getsize(
        os.path.join(OUTPUT_DIR, f)
    )
    print(f"{f} | {size:,} bytes")

print()
print("NUMBER OF CSV FILES:", len(files))
print("=" * 70)

LAST SAVED ID: 898049

CSV FILES:
batch_00001.csv | 1,157,922 bytes
batch_00002.csv | 1,159,580 bytes
batch_00003.csv | 1,148,954 bytes
batch_00004.csv | 1,152,614 bytes
batch_00005.csv | 1,134,295 bytes
batch_00006.csv | 1,137,310 bytes
batch_00007.csv | 1,127,948 bytes
batch_00008.csv | 1,147,565 bytes
batch_00009.csv | 1,154,817 bytes
batch_00010.csv | 1,129,298 bytes
batch_00011.csv | 1,148,272 bytes
batch_00012.csv | 1,148,589 bytes
batch_00013.csv | 1,137,951 bytes
batch_00014.csv | 1,129,862 bytes
batch_00015.csv | 1,125,596 bytes
batch_00016.csv | 1,128,515 bytes
batch_00017.csv | 1,133,724 bytes
batch_00018.csv | 1,137,720 bytes
batch_00019.csv | 1,139,072 bytes
batch_00020.csv | 1,126,025 bytes
batch_00021.csv | 1,138,539 bytes
batch_00022.csv | 1,131,440 bytes
batch_00023.csv | 1,128,544 bytes
batch_00024.csv | 1,126,597 bytes
batch_00025.csv | 1,118,264 bytes
batch_00026.csv | 1,124,270 bytes
batch_00027.csv | 1,134,740 bytes
batch_00028.csv | 1,129,243 bytes
batch_00029.cs